# Project 4 : De Bruijn graphs

## Imports and utlilities

In [1]:
# Imports
import gzip
import random
import sys
import time
from datetime import datetime
from collections import defaultdict

# Set recursion depth for large genome
sys.setrecursionlimit(1000000)

def read_fastq(filename):
    sequences = []
    with gzip.open(filename, 'rt') as f:
        line_count = 0
        for line in f:
            line_count += 1
            if line_count % 4 == 2: # Sequence line in FASTQ format
                sequences.append(line.strip())
    return sequences

print("✅ Imports and read_fastq function loaded successfully.")

# For subsample
reads = read_fastq("data/mouse_SE_150bp.fq.gz")
sample = random.sample(reads, 8000)


✅ Imports and read_fastq function loaded successfully.


## The DeBruijnGraph Class

In [2]:
class DeBruijnGraph:
    def __init__(self, reads, k):
        self.graph = defaultdict(list)
        self.k = k
        self.build_graph_from_reads(reads, k)

    def add_edge(self, left, right):
        self.graph[left].append(right)

    def remove_edge(self, left, right):
        if right in self.graph[left]:
            self.graph[left].remove(right)

    def build_graph_from_reads(self, reads, k):
        for read in reads:
            for i in range(len(read) - k + 1):
                k_mer = read[i: i + k]
                l_mer = k_mer[0:k-1]
                r_mer = k_mer[1:]
                self.add_edge(l_mer, r_mer)

    def eulerian_walk(self, node, graph):
        tour = []
        while graph[node]:
            next_node = random.choice(graph[node])
            graph[node].remove(next_node)
            # This is the line that handles the recursion
            tour.extend(self.eulerian_walk(next_node, graph))
        return tour + [node]

    def assemble_contigs(self, seed=None):
        if seed is not None:
            random.seed(seed)

        start_nodes = []
        target_nodes = {v for targets in self.graph.values() for v in targets}
        start_nodes = [node for node in self.graph.keys() if node not in target_nodes]

        # If start nodes is empty use the first key
        if not start_nodes:
            start_nodes = [list(self.graph.keys())[0]]

        contigs = []
        graph_copy = defaultdict(list, {k: v[:] for k, v in self.graph.items()})
        for node in start_nodes:
            if graph_copy[node]:
                walk = self.eulerian_walk(node, graph_copy)
                # Reverse order of the walk and assemble sequences
                contig = self.tour_to_sequence(walk[::-1])
                contigs.append(contig)
        return contigs

    def tour_to_sequence(self, tour):
        if not tour: return ""
        # Start with the first (k-1)-mer
        seq = tour[0]

        # Append only the last base of each node
        for k1_mer in tour[1:]:
            seq += k1_mer[-1]
        return seq

    def get_assembly_stats(self, contigs):
        """
        Calculate assembly statistics for assembled contigs
        """
        stats = {
            "num_contigs": 0,
            "total_length": 0,
            "longest_contig": 0,
            "shortest_contig": 0,
            "mean_length": 0,
            "n50": 0
        }
        if not contigs:
            return stats

        # List of lengths of each contig
        lengths = [len(contig) for contig in contigs]

        # Calculate n50
        sorted_lengths = sorted(lengths, reverse=True)
        cumulative_length = 0
        half_total = sum(lengths)/2
        n50 = 0
        for length in sorted_lengths:
            cumulative_length += length
            if cumulative_length >= half_total:
                n50 = length
                break

        stats["num_contigs"] = len(lengths)
        stats["total_length"] = sum(lengths)
        stats["longest_contig"] = max(lengths)
        stats["shortest_contig"] = min(lengths)
        stats["mean_length"] = (sum(lengths)/len(lengths))
        stats["n50"] = n50

        return stats

    def write_fasta(self, contigs, filename):
        """
        Write assembled contigs to a FASTA file.
        """
        with open (filename, 'w') as out:
            for i, contig in enumerate(contigs, start=1):
                out.write(f'>Contig_{i}\n')
                out.write(f'{contig}\n')

print("✅DeBruijnGraph Class defined successfully.")

✅DeBruijnGraph Class defined successfully.


---
# Toy Example

In [3]:
# Toy Data Definition
toy_reads_1 = [
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA"   # Read 9: overlaps with Read 8
]

print("="*60)
print("ASSEMBLING TOY EXAMPLE")
print("="*60)

# Build graph with k=6 and use a random seed for reproducibility
dbg_toy = DeBruijnGraph(toy_reads_1, k=6)
toy_result = dbg_toy.assemble_contigs(seed=42)

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=6...")

if toy_result:
    print(f"Sequence: {toy_result[0]}")
    print(f"Length:   {len(toy_result[0])} bp ")
    print("\n✅ Toy Example assembled successfully.")
else:
    print("❌ Toy Example failed to assemble.")

ASSEMBLING TOY EXAMPLE

Input: 9 reads
First read:   ATGGCGTACG
Last read:    TGGGCCTAAA

Building De Bruijn graph with k=6...
Sequence: ATGGCGTAACCGGGTTTTTAACCCCCAAATTGGGGGGGGCCCCCTTAAAA
Length:   50 bp 

✅ Toy Example assembled successfully.


---
# Subsample of data

In [4]:
# DBG example with just a small subset of reads, in our current implementation
# We have not gotten the full set of reads to work
# We run into memory errors along with recursion limits with the current implementation, so the driver program won't work as intended
# It does work on a smaller subset of the samples that we previously selected at random


dbg = DeBruijnGraph(sample, k = 51)
contigs = dbg.assemble_contigs(seed=7)
stats = dbg.get_assembly_stats(contigs)
    
print(f"Assembly Statistics:")
print(f"  Number of contigs:     {stats['num_contigs']:,}")
print(f"  Total assembly length: {stats['total_length']:,} bp")
print(f"  Longest contig:        {stats['longest_contig']:,} bp")
print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
print(f"  N50:                   {stats['n50']:,} bp")
print()

dbg.write_fasta(contigs, "subsample_assembly.fa")

Assembly Statistics:
  Number of contigs:     7,857
  Total assembly length: 1,191,950 bp
  Longest contig:        906 bp
  Shortest contig:       51 bp
  Mean contig length:    151.7 bp
  N50:                   150 bp



---
# Real data
This section utilizes a FASTQ file with 10 million "perfect" 150 bp reads simulated
from the mouse genome (`GRCm39`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [ ]:
from datetime import datetime


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    random_seed,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    coverage_estimate,
    assembly_fraction
):
    """Write comprehensive assembly statistics to text file.
    """
    with open(stats_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE GENOME ASSEMBLY STATISTICS\n")
        f.write("="*80 + "\n\n")

        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input File: {input_file}\n")
        f.write(f"K-mer Size: {k_mer_size}\n")
        f.write(f"Random Seed: {random_seed}\n\n")

        f.write("-"*80 + "\n")
        f.write("INPUT DATA\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")

        f.write("-"*80 + "\n")
        f.write("DE BRUIJN GRAPH CONSTRUCTION\n")
        f.write("-"*80 + "\n")
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        f.write(f"Average out-degree:      {num_edges/num_nodes:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")

        f.write("-"*80 + "\n")
        f.write("ASSEMBLY RESULTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")

        f.write("-"*80 + "\n")
        f.write("TOP 20 LONGEST CONTIGS\n")
        f.write("-"*80 + "\n")
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")

        f.write("-"*80 + "\n")
        f.write("CONTIG LENGTH DISTRIBUTION\n")
        f.write("-"*80 + "\n")
        bins = [
            (">100kb", sum(1 for x in contig_lengths if x > 100000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")

        f.write("-"*80 + "\n")
        f.write("TIMING SUMMARY\n")
        f.write("-"*80 + "\n")
        total_time = timing['total_time']
        f.write(f"Read time:               {timing['read_time']:8.2f} seconds "
                f"({timing['read_time']/total_time*100:5.1f}%)\n")
        f.write(f"Graph construction:      {timing['graph_time']:8.2f} seconds "
                f"({timing['graph_time']/total_time*100:5.1f}%)\n")
        f.write(f"Assembly:                {timing['assembly_time']:8.2f} seconds "
                f"({timing['assembly_time']/total_time*100:5.1f}%)\n")
        f.write(f"Total time:              {total_time:8.2f} seconds "
                f"({total_time/60:.2f} minutes)\n\n")

        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")


def assemble_mouse_genome(
    input_file="data/mouse_SE_150bp.fq",
    output_fasta="mouse_assembly.fasta",
    stats_file="mouse_assembly_stats.txt",
    k_mer_size = 51,
    random_seed = None
):
    """Main driver function for mouse genome assembly."""
    print("="*80)
    print("MOUSE GENOME ASSEMBLY PIPELINE")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()

    timing = {}
    print("STEP 1: Loading sequencing reads")
    print("-"*80)
    print(f"Input file: {input_file}")
    print(f"Expected: 10 million reads, 150bp each")
    print()

    start_time = time.time()
    reads = read_fastq(input_file)
    timing['read_time'] = time.time() - start_time

    num_reads = len(reads)
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads if num_reads > 0 else 0

    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")

    if num_reads > 0:
        print(f"\nSample reads:")
        print(f"  First: {reads[0][:80]}...")
        print(f"  Last:  {reads[-1][:80]}...")
    print()

    print("STEP 2: Building De Bruijn graph")
    print("-"*80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...")
    print()

    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing['graph_time'] = time.time() - start_time

    # Calculate graph statistics
    num_nodes = len(dbg.graph)
    num_edges = sum(len(neighbors) for neighbors in dbg.graph.values())
    avg_degree = num_edges / num_nodes if num_nodes > 0 else 0

    print(f"Graph construction complete!")
    print(f"  Nodes (unique {k_mer_size-1}-mers): {num_nodes:,}")
    print(f"  Edges (k-mer transitions): {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds")
    print()

    print("STEP 3: Assembling contigs")
    print("-"*80)
    print(f"Finding connected components and traversing graph...")
    print(f"Random seed: {random_seed} (for reproducibility)")
    print()

    start_time = time.time()
    contigs = dbg.assemble_contigs(seed=random_seed)
    timing['assembly_time'] = time.time() - start_time

    print(f"Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds")
    print()

    print("STEP 4: Calculating assembly statistics")
    print("-"*80)

    stats = dbg.get_assembly_stats(contigs)

    print(f"Assembly Statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp")
    print()

    # Display distribution of contig lengths
    contig_lengths = sorted([len(c) for c in contigs], reverse=True)

    print(f"Contig Length Distribution:")
    print(f"  Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"    {i:2d}. {length:,} bp")
    print()

    # Calculate coverage estimate
    genome_size_estimate = 2700000000  # Mouse genome ~2.7 Gbp
    coverage_estimate = (num_reads * avg_read_length) / genome_size_estimate
    assembly_fraction = (stats['total_length'] / genome_size_estimate) * 100

    print(f"Genome Coverage Analysis:")
    print(f"  Mouse genome size (expected): ~{genome_size_estimate:,} bp")
    print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
    print(f"  Assembly size vs. genome: {assembly_fraction:.1f}%")
    print()

    print("STEP 5: Writing output files")
    print("-"*80)

    # Write assembled contigs to FASTA
    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")

    # Write detailed statistics
    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        random_seed,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        coverage_estimate,
        assembly_fraction
    )
    print(f"✓ Statistics written to: {stats_file}")
    print()

    timing['total_time'] = (timing['read_time'] + timing['graph_time'] +
                           timing['assembly_time'])

    print("="*80)
    print("ASSEMBLY COMPLETE")
    print("="*80)
    print(f"Total time: {timing['total_time']:.2f} seconds "
          f"({timing['total_time']/60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()

    print(f"Summary:")
    print(f"  • Processed {num_reads:,} reads ({total_bases:,} bp)")
    print(f"  • Built graph with {num_nodes:,} nodes and {num_edges:,} edges")
    print(f"  • Assembled {stats['num_contigs']:,} contigs")
    print(f"  • Total assembly: {stats['total_length']:,} bp (N50: {stats['n50']:,} bp)")
    print(f"  • Output files: {output_fasta}, {stats_file}")
    print()

    return {
        'dbg': dbg,
        'contigs': contigs,
        'stats': stats,
        'timing': timing,
        'graph_stats': {
            'num_nodes': num_nodes,
            'num_edges': num_edges,
            'avg_degree': avg_degree
        },
        'coverage': coverage_estimate
    }


print("\n" + "="*80)
print("MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("="*80 + "\n")

# Run the assembly pipeline
result = assemble_mouse_genome(input_file="data/mouse_SE_150bp.fq.gz")

# Display sample contigs
print("="*80)
print("SAMPLE ASSEMBLED CONTIGS")
print("="*80 + "\n")

contigs = result['contigs']
for i in range(min(5, len(contigs))):
    contig = contigs[i]
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i+1} length={len(contig)}")
    print(preview)
    print()

print("="*80)
print("✓ ASSEMBLY COMPLETE")
print("="*80)
print(f"\nResults saved to:")
print(f"  • mouse_assembly.fasta - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • mouse_assembly_stats.txt - Detailed assembly statistics")
print(f"\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
print(f"  • Coverage: {result['coverage']:.1f}x")
print()



MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

MOUSE GENOME ASSEMBLY PIPELINE
Start time: 2026-02-25 05:51:01

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: data/mouse_SE_150bp.fq.gz
Expected: 10 million reads, 150bp each

Reads loaded: 10,000,001
Total bases: 1,500,000,150 bp
Average read length: 150.0 bp
Time elapsed: 13.29 seconds

Sample reads:
  First: AATCAGGGAGAGACTGGGAGAAGTGGAGGGAGTGGAAATCACAGGAGGGATGTAATATATGAGAGAAGAATAAATGTAAG...
  Last:  TTTAGGCATTCCGGTGTTGGGTTAACAGAGAAGTTATAGGTGGATTATTTATAGTGTGATTATTGCCTATAGTCTGATTA...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 51
Building graph from 10,000,001 reads...

